# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Key design point: the 11.9k rows come from only ~1,500 people (~8 stays each). Rows from the same person are not independent, so **all validation here is grouped by `person_key`**; a random row split would let the model see the same person in train and test. We also show how much that would inflate results.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
SEED=0
df=pd.read_csv('data.csv')
df.shape, df.isna().sum().sum(), df.duplicated().sum()

## 1. Data checks

In [ ]:
print(df.describe().T.round(2))
print('people:',df.person_key.nunique(),' stays/person:',df.groupby('person_key').size().describe()[['min','mean','max']].round(1).to_dict())
print('readmission rate:',df.readmit_30d.mean().round(4))

In [ ]:
feats=['visit_number','birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b','postcode_area']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,f in zip(ax.ravel(),feats):
    q=pd.qcut(df[f],10,duplicates='drop')
    r=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey'); a.set_title(f); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 2. Modelling setup
Features: everything except `person_key` (an identifier, not a signal). `sex` is 0/1; `postcode_area` is a numeric area code, so it is one-hot encoded for the linear model and left numeric for the tree model.
Metrics: ROC AUC, average precision (baseline = prevalence ~0.20), Brier score. 5-fold `StratifiedGroupKFold`.

In [ ]:
X=df.drop(columns=['person_key','readmit_30d']).copy(); X['sex']=(X.sex=='M').astype(int); y=df.readmit_30d.values; g=df.person_key.values
num=[c for c in X if c!='postcode_area']
lr=make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('p',OneHotEncoder(handle_unknown='ignore'),['postcode_area'])]),LogisticRegression(max_iter=2000,C=0.1))
hgb=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,min_samples_leaf=50,l2_regularization=1.0,random_state=SEED)
models={'baseline (prevalence)':DummyClassifier(strategy='prior'),'logistic regression':lr,'gradient boosting':hgb}
cv=StratifiedGroupKFold(5,shuffle=True,random_state=SEED)
def evaluate(cv_,groups,yy=y):
    out,preds={}, {}
    for n,m in models.items():
        p=np.zeros(len(yy)); aucs=[]
        for tr,te in cv_.split(X,yy,groups):
            m.fit(X.iloc[tr],yy[tr]); p[te]=m.predict_proba(X.iloc[te])[:,1]; aucs.append(roc_auc_score(yy[te],p[te]))
        out[n]=dict(AUC=roc_auc_score(yy,p),AUC_fold_sd=np.std(aucs),AP=average_precision_score(yy,p),Brier=brier_score_loss(yy,p)); preds[n]=p
    return pd.DataFrame(out).T, preds
res,preds=evaluate(cv,g); res.round(4)

## 3. Is the result real? Leakage check and a label-permutation test

In [ ]:
# (a) Naive row-level CV ignoring person -> shows how much person-level leakage would inflate results
naive,_=evaluate(KFold(5,shuffle=True,random_state=SEED),None)
print('Naive row-level CV (leaky):'); print(naive.round(4))
# (b) Null distribution: shuffle labels, rerun grouped CV for logistic regression
rng=np.random.default_rng(SEED); null=[]
for _ in range(20):
    yy=rng.permutation(y); p=np.zeros(len(y))
    for tr,te in cv.split(X,yy,g):
        lr.fit(X.iloc[tr],yy[tr]); p[te]=lr.predict_proba(X.iloc[te])[:,1]
    null.append(roc_auc_score(yy,p))
print('Null AUC (shuffled labels): mean %.3f, max %.3f   vs observed LR %.3f'%(np.mean(null),np.max(null),res.loc['logistic regression','AUC']))

## 4. Performance plots and calibration

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(17,5))
for n,p in preds.items():
    fpr,tpr,_=roc_curve(y,p); ax[0].plot(fpr,tpr,label=f'{n} ({roc_auc_score(y,p):.3f})')
    pr,rc,_=precision_recall_curve(y,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_title('ROC (grouped CV)'); ax[0].legend()
ax[1].axhline(y.mean(),c='k',ls='--'); ax[1].set_title('Precision-recall'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision')
for n in ['logistic regression','gradient boosting']:
    b=pd.qcut(preds[n],10,duplicates='drop'); t=pd.DataFrame({'p':preds[n],'y':y}).groupby(b,observed=True).mean()
    ax[2].plot(t.p,t.y,'o-',label=n)
ax[2].plot([0,.6],[0,.6],'k--'); ax[2].set_title('Calibration (deciles)'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed'); ax[2].legend()
plt.tight_layout(); plt.show()
for n in ['logistic regression','gradient boosting']:
    p=preds[n]; k=len(y)//10; o=np.argsort(-p)
    print('%s: top-decile readmit rate %.3f vs overall %.3f (lift %.2fx)'%(n,y[o[:k]].mean(),y.mean(),y[o[:k]].mean()/y.mean()))

## 5. What drives the prediction?

In [ ]:
imp=[]
for tr,te in cv.split(X,y,g):
    hgb.fit(X.iloc[tr],y[tr]); r=permutation_importance(hgb,X.iloc[te],y[te],scoring='roc_auc',n_repeats=10,random_state=SEED); imp.append(r.importances_mean)
pd.Series(np.mean(imp,0),X.columns).sort_values().plot.barh(title='Permutation importance (drop in AUC, held-out folds)'); plt.show()
lr.fit(X,y); print('Standardised logistic coefficients:'); print(pd.Series(lr[-1].coef_[0][:len(num)],num).round(3).sort_values())

## 6. Conclusions (from a run with SEED=0)
- **Signal exists but is modest.** Grouped 5-fold CV: ROC AUC ≈ 0.63 (logistic) / 0.64 (gradient boosting) vs 0.50 for the baseline; average precision ≈ 0.30 vs 0.20 prevalence. Shuffled-label AUCs stay at 0.48–0.51, so this is not noise.
- **Useful for triage, not for individual decisions.** The top-risk decile is readmitted ≈ 37% of the time vs 20% overall (≈1.9× lift).
- **Drivers:** length of stay, lab A and number of medications. Age, lab B, BMI, sex, visit number and postcode add almost nothing. Logistic and boosting perform about equally, so the relationship is close to linear.
- **Leakage matters.** A naive row-level split gives AUC ≈ 0.66–0.68, an overstatement caused by the same people appearing in train and test. Use grouped validation.
- **Caveats:** one dataset, no external or temporal validation; the true held-out AUC may differ. The fold SD is ~0.02. Check calibration before using the probabilities as risks.